In [7]:
import re
import random
import numpy as np
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from datasets import load_dataset
from datasets import concatenate_datasets
from sentence_transformers import SentenceTransformer

In [8]:
ds = load_dataset("rajpurkar/squad")
ds.column_names

{'train': ['id', 'title', 'context', 'question', 'answers'],
 'validation': ['id', 'title', 'context', 'question', 'answers']}

In [9]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
splitter = RecursiveCharacterTextSplitter(chunk_size=250,chunk_overlap=50)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3861.94it/s]


In [10]:
all_contexts = list(ds['train']['context']) + list(ds['validation']['context'])
all_query = list(ds['train']['question']) + list(ds['validation']['question'])

In [20]:
def add_label(example,label):
    example['label'] = label
    return example


def jaccard(q, c):
    q = set(q.lower().split())
    c = set(c.lower().split())
    return len(q & c) / len(q | c)


def query_coverage(q, c):
    q = set(q.lower().split())
    c = set(c.lower().split())
    return len(q & c) / len(q)


def shared_tokens(q, c):
    return len(
            set(q.lower().split()) &
            set(c.lower().split()))


def add_features(example):
    query = example['question']
    context = example['context']

    example['jaccard']=jaccard(query,context)
    example['context_length']= len(example['context'])
    example['query_coverage']=query_coverage(query,context)
    example['shared_tokens']=shared_tokens(query,context)

    return example

def swap_context(example):
    context = random.choice(all_contexts)
    while context == example["context"]:
        context = random.choice(all_contexts)

    example["context"] = context
    example["label"]=0
    
    return example


def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1)
    vec2 = np.asarray(vec2)

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return np.dot(vec1, vec2)/(norm1 * norm2)

In [12]:
all_query_embeddings = model.encode(all_query)
all_chunks = []

for context in all_contexts:
    chunks = splitter.split_text(context)
    all_chunks.extend(chunks)

all_chunk_embeddings = model.encode(all_chunks)

query_cache = dict(zip(all_query,all_query_embeddings))
chunk_cache = dict(zip(all_chunks, all_chunk_embeddings))

In [13]:
def remove_best_chunk(example):
    context = example["context"]
    query = example["question"]

    chunks = splitter.split_text(context)
    if len(chunks) <= 1:
        return example

    query_emb = query_cache[query] if query in query_cache else model.encode(query)
    chunk_embeddings = [chunk_cache[chunk] if chunk in chunk_cache else model.encode(chunk) for chunk in chunks]


    similarities = [cosine_similarity(query_emb, ce) for ce in chunk_embeddings]
    best_idx = np.argmax(similarities)

    false_context = random.choice(all_contexts)
    while false_context == context:
        false_context = random.choice(all_contexts)
    
    false_chunks = splitter.split_text(false_context)

    if false_chunks:
        chunks[best_idx] = random.choice(false_chunks)

    example["context"] = " ".join(chunks)
    example["label"] = 0
    return example

In [14]:
ds = ds.remove_columns(["id", "title", "answers"])

In [15]:
ds["train"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'}

In [16]:
ds = ds.map(lambda example: add_label(example, label=1))

In [17]:
negative_context_ds = ds.map(swap_context)
negative_context_ds_2 = ds.map(remove_best_chunk)

half = len(ds['train']) // 2

neg1 = negative_context_ds['train'].select(range(half))
neg2 = negative_context_ds_2['train'].select(range(len(ds['train']) - half))

half = len(ds['validation']) // 2

vneg1 = negative_context_ds['validation'].select(range(half))
vneg2 = negative_context_ds_2['validation'].select(range(len(ds['validation']) - half))

In [18]:
final_ds = concatenate_datasets([ds['train'], neg1, neg2,vneg1,vneg2]).shuffle(seed=42)

In [21]:
final_ds = final_ds.map(add_features)

Map: 100%|██████████| 185768/185768 [01:05<00:00, 2823.09 examples/s]


In [22]:
final_ds = final_ds.shuffle(seed=50)
final_ds

Dataset({
    features: ['context', 'question', 'label', 'jaccard', 'context_length', 'query_coverage', 'shared_tokens'],
    num_rows: 185768
})

In [23]:
df = final_ds.to_pandas()
df.to_csv('../datasets/SQuAD_processed.csv', index=False)

In [26]:
df.sample(5)

,context,question,label,jaccard,context_length,query_coverage,shared_tokens
115334,Season six premiered with the series' highest-...,Which television network originally aired The ...,0,0.024793,1127,0.375000,3
175282,The polarization of an antenna refers to the o...,When is a magnetic fields right angles to a el...,1,0.106061,677,0.700000,7
179203,Diagnosis of infectious disease sometimes invo...,How are minor infectious diseases treated?,1,0.045455,839,0.666667,4
142203,Muslim scientists placed far greater emphasis ...,What famous ship left Southampton's port carry...,0,0.010638,838,0.100000,1
137061,Buddhism /ˈbudɪzəm/ is a nontheistic religion[...,Who's teachings is Buddhism based upon?,1,0.043478,805,0.666667,4
